In [1]:
# Getting imports and the model builder
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")
sys.path.insert(0, "../baseline")

from smolagents import OpenAIModel
from dotenv import load_dotenv
load_dotenv()

MODEL_NAMES = ["gpt-4o", "gpt-5.4-mini", "Qwen/Qwen3.7-Plus", "Qwen/Qwen3.5-9B"]
WINDOW_SIZES = [1, 3, 5]


def build_model(model_name: str) -> OpenAIModel:
    if model_name in ["gpt-4o", "gpt-5.4-mini"]:
        return OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
    # Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served
    # open-weight models -- verified this works for both Qwen3.7-Plus and Qwen3.5-9B.
    # client_kwargs timeout bounds a single API call so a stalled/hanging stream fails within
    # 5 minutes instead of hanging indefinitely -- evaluate_agent already catches such errors.
    return OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        client_kwargs={"timeout": 300.0},
    )

In [2]:
# Getting the tools setup and the MarkovReActCodeAgent builder
from smolagents.monitoring import LogLevel
from common_setup import assert_no_reasoning, build_tools
from smolagents.markov_react import MarkovReActCodeAgent


def build_agent(model, model_name: str, window_size: int, tools):
    return MarkovReActCodeAgent(
        tools=tools,
        model=model,
        max_steps=50,
        verbosity_level=LogLevel.DEBUG,
        additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
        window_size=window_size,
        stream_outputs="Qwen" in model_name,
        step_callbacks=[assert_no_reasoning] if "Qwen" in model_name else None,
    )

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Fermi (RealFP) dataset

Same RealFP split, `fermi_scorer`, and `FERMI_ANSWER_INSTRUCTION` used in
[naiveReAct_fermi.ipynb](../baseline/naiveReAct_fermi.ipynb), reused here so MarkovReAct stays
directly comparable to naive ReAct on Fermi -- same dataset, same scorer, same prompt suffix.

In [3]:
# Load the Fermi RealFP validation split
import pandas as pd
from common_setup import evaluate_agent
from fermi_setup import load_fermi_dataset, fermi_scorer, FERMI_ANSWER_INSTRUCTION

eval_ds = load_fermi_dataset(split="val")
print(f"Loaded {len(eval_ds)} examples")

Loaded 125 examples


In [4]:
# One model (and its tools) is built once, then reused across all 3 window sizes.
all_results = {}

for model_name in MODEL_NAMES:
    model = build_model(model_name)
    tools, ti_tool, visualizer = build_tools(model)

    for window_size in WINDOW_SIZES:
        agent = build_agent(model, model_name, window_size, tools)
        safe_model_name = model_name.replace("/", "_")
        pickle_dir = f"markov_fermi_{safe_model_name}_w{window_size}"
        print(f"\n=== {model_name}, window_size={window_size} ===")
        results = evaluate_agent(
            agent,
            eval_ds,
            ti_tool,
            visualizer,
            output_file=f"{pickle_dir}.jsonl",
            pickle_dir=pickle_dir,
            scorer=fermi_scorer,
            question_suffix=FERMI_ANSWER_INSTRUCTION,
        )
        all_results[(model_name, window_size)] = results


=== gpt-4o, window_size=1 ===

[1/125] (cached) If all but 1 million people on Earth died, how far (on average) would you have to walk to meet someo...
  ✓ | cached

[2/125] (cached) How much water would need to be evaporated to turn Arizona tropical?...
  ✗ | cached

[3/125] (cached) If a gallon of paint is used to coat 400 ft2 of walls, how thick, cm, is the paint film?...
  ✓ | cached

[4/125] (cached) How much space would be required to fit 1 individual of each species of plants and animals?...
  ✓ | cached

[5/125] (cached) How many canaries in the world are currently flying?...
  ✓ | cached

[6/125] (cached) How old are you if you are a million hours old?...
  ✓ | cached

[7/125] (cached) How many mice could pull the Maus tank?...
  ✓ | cached

[8/125] (cached) How many times does your heart beat per week?...
  ✓ | cached

[9/125] (cached) What do you estimate the total weight of all the Covid-19 viruses in the world?...
  ✓ | cached

[10/125] (cached) How many people are airbor


[31/125] (cached) How many nerd ropes would it take to go around the Earth?...
  ✓ | cached

[32/125] (cached) How many golf balls could you fit in a 5 gallon water cooler jug?...
  ✓ | cached

[33/125] (cached) How many cells are needed to visible to a person?...
  ✓ | cached

[34/125] (cached) How much does it cost to leave a light on for an entire year?...
  ✓ | cached

[35/125] (cached) How many model rockets would it take to propel a 140 lb human?...
  ✓ | cached

[36/125] (cached) The Ontario government decides that winter is too depressing and that what is really needed is to tu...
  ✗ | cached

[37/125] (cached) How much tire rubber, in lbs, is shed by normal wear and tear of car tires in the USA each year?...
  ✓ | cached

[38/125] (cached) What is the total length of waterslides in the United States?...
  ✗ | cached

[39/125] (cached) In a typical cup of brewed tea with no milk, sugar and with leaves strained or tea bag taken out wha...
  ✓ | cached

[40/125] (cached) What i

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ The Ontario government decides that winter is too depressing and that what is really needed is to turn all of   │
│ Lake Ontario into one giant mug of hot chocolate. Find the number of packets of hot chocolate powder required   │
│ to make this much hot chocolate.                                                                                │
│                                                                                                                 │
│ Give your final answer as a number with units matching the question (e.g. "22.3 km", "79 g"; use a bare number  │
│ like "33000" only if the question has no natural unit). Do not include filler words like "about" or "roughly",  │
│ or thousands separators. Write the full number of digits (e.g. "256000000000") rather than word multipliers     │
│ like "million" or "billion".                                                                                    │
│                                                                                                                 │
╰─ OpenAIModel - Qwen/Qwen3.7-Plus ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output()

...

Execution logs:
PMC estimate: 3400000000 lbs

Final answer: 3400000000

[Step 13: Duration 8.65 seconds| Input tokens: 96,273 | Output tokens: 2,092]

  ✗ | 69.4s | 13 steps | tokens: TokenUsage(input_tokens=96273, output_tokens=2092, total_tokens=98365)
    Expected: 2.64 lbs
    Got:      3400000000

[38/125] (cached) What is the total length of waterslides in the United States?...
  ✗ | cached

[39/125] (cached) In a typical cup of brewed tea with no milk, sugar and with leaves strained or tea bag taken out wha...
  ✓ | cached

[40/125] (cached) What is the total number of shots taken in one NBA season, including the tournament?...
  ✗ | cached

[41/125] (cached) What is the average number of bricks used to build a building?...
  ✓ | cached

[42/125] (cached) How many sheets of paper could be stacked from the floor to the ceiling?...
  ✓ | cached

[43/125] (cached) How many metres of spaghetti would be needed if all of the children in a school had spaghetti bologn...
  ✓ | cached

[44/125] (cached) How many pennies are in a typical jar?...
  ✓ | cached

[45/125] (cached) What is the limit of solid planet size?...
  ✗ | cached

[46


[116/125] (cached) How many softballs are produced annually?...
  ✓ | cached

[117/125] (cached) For the United States, how many cars would have their fuel needs met for one year by the fat contain...
  ✓ | cached

[118/125] (cached) How long does it take for a human body to burn at the stake?...
  ✓ | cached

[119/125] (cached) What is the mass of air in a room?...
  ✓ | cached

[120/125] (cached) How many illiterate people are there in India?...
  ✓ | cached

[121/125] (cached) How many graduates, people holding first university degrees, do we produce in our country?...
  ✓ | cached

[122/125] (cached) How many gallons of water in the Atlantic ocean?...
  ✓ | cached

[123/125] (cached) How much electricity would one need to stop a car from moving forward?...
  ✗ | cached

[124/125] (cached) If the top 10 Forbes businesses donated 10% of their annual proceeds to schools, how much money woul...
  ✗ | cached

[125/125] (cached) How much air would it take to fill all of the school's bas

In [5]:
rows = []
for (model_name, window_size), results in all_results.items():
    df = pd.DataFrame(results)
    total = len(df)
    avg_score = df["is_correct"].mean()  # continuous 0-1 order-of-magnitude accuracy metric, not a boolean fraction
    total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0))
    rows.append({
        "Model": model_name,
        "window_size": window_size,
        "Avg Score": round(avg_score, 3),
        "Avg Steps": round(df["num_steps"].mean(), 1),
        "Avg Tokens": round(total_tokens.mean()),
    })

pd.DataFrame(rows)

,Model,window_size,Avg Score,Avg Steps,Avg Tokens
0,gpt-4o,1,0.595,4.8,17907
1,gpt-4o,3,0.556,4.2,20806
2,gpt-4o,5,0.579,4.5,25299
3,gpt-5.4-mini,1,0.425,2.2,6408
4,gpt-5.4-mini,3,0.451,2.2,6443
5,gpt-5.4-mini,5,0.392,2.2,6553
6,Qwen/Qwen3.7-Plus,1,0.515,21.7,92461
7,Qwen/Qwen3.7-Plus,3,0.551,20.6,130371
8,Qwen/Qwen3.7-Plus,5,0.554,17.8,142999
9,Qwen/Qwen3.5-9B,1,0.546,21.0,88980
